In [0]:
# Catalog -> Schema -> Table
# Catalog -> Schema -> Volume -> File 

catalog = "coldchain_lakehouse"

volume = f"/Volumes/{catalog}/bronze/landing_zone"

sensor_telemetry_file = f"{volume}/sensor_telemetry.csv"
shipments_file = f"{volume}/shipments.csv"
device_registry_file = f"{volume}/device_registry.csv"

bronze_sensor_telemetry_table = f"{catalog}.bronze.sensor_telemetry"
bronze_shipments_table = f"{catalog}.bronze.shipement"
bronze_device_registry_table = f"{catalog}.bronze.device_registry"

In [0]:
from pyspark.sql.functions import col,current_timestamp

In [0]:
df_sensor_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(sensor_telemetry_file)
)

In [0]:
df_shipments_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(shipments_file)
)

In [0]:
df_device_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(device_registry_file)
)

In [0]:
df_sensor_raw.printSchema()
df_shipments_raw.printSchema()
df_device_raw.printSchema()

### Insert Timestamp col and Source File col

In [0]:
df_sensor_bronze = (
    df_sensor_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
df_shipments_bronze = (
    df_shipments_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
df_device_bronze = (
    df_device_raw
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path"))
)

In [0]:
display(df_sensor_bronze)

In [0]:
df_shipments_bronze.display()

In [0]:
display(df_device_bronze)

### Bronze DataFrame → convert it into Delta format → append the data → save it as a table in Databricks.

In [0]:
(
    df_sensor_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_sensor_telemetry_table)
)

In [0]:
(
    df_shipments_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_shipments_table)
)

In [0]:
(
    df_device_bronze.write
    .format("delta")
    .mode("append")
    .saveAsTable(bronze_device_registry_table)
)

In [0]:
print("Tables Loaded Successfully")
print(bronze_sensor_telemetry_table)
print(bronze_shipments_table)
print(bronze_device_registry_table)

In [0]:
print("\nRow Count")
print(f"{bronze_sensor_telemetry_table} : {spark.table(bronze_sensor_telemetry_table).count()}")
print(f"{bronze_shipments_table} : {spark.table(bronze_shipments_table).count()}")
print(f"{bronze_device_registry_table} : {spark.table(bronze_device_registry_table).count()}")

In [0]:
print("Sensor Telemetry Schema")
spark.table(bronze_sensor_telemetry_table).printSchema()

In [0]:
print("Shipement Table Schema")
spark.table(bronze_shipments_table).printSchema()

In [0]:
print("Device Registry Table Schema")
spark.table(bronze_device_registry_table).printSchema()

In [0]:
display(
    spark.table(bronze_sensor_telemetry_table)
    .select("_ingestion_timestamp", "_source_file")
    .distinct()
)

In [0]:
display(
    spark.table(bronze_shipments_table)
    .select("_ingestion_timestamp", "_source_file")
    .distinct()
)

In [0]:
display(
    spark.table(bronze_device_registry_table)
    .select("_ingestion_timestamp", "_source_file")
    .distinct()
)